In [1]:
import pandas as pd 
import numpy as np
import json
import os

In [ ]:
run_TrimerSQL = True

if run_TrimerSQL:
    import scripts.TrimerSQL as sql
    
    # Setting up database connection via json config file
    with open("sql.json") as file:
        sql_info = json.load(file)
    
    username, password, ip, port, database = list(sql_info.values())

    # SQL conneccot, the info_list need to contain the host ip, port, username, password and wasted database
    info_list = [ip, port, username, password, database]

    sql_conn = sql.mysql(input_host = info_list[0],
                         input_port = info_list[1],
                         input_username = info_list[2],
                         input_password = info_list[3],
                         input_database = info_list[4])
    
    sql_conn.import_tables()

In [2]:
run_TrimerCreate = True

if run_TrimerCreate:
    import scripts.TrimerCreate as trimer
    source_options = ["germline", "top_seq", "all_seq", "representative"][-1]
    trim_obj = trimer.Trimer(seqs_loc="trimers_raw_tables\\covid_vaccine_new.sequences.csv",
                             seq_collapsed_loc="trimers_raw_tables\\covid_vaccine_new.sequence_collapse.csv", 
                             metadata_loc="trimers_raw_tables\\covid_vaccine_new.sample_metadata.csv",
                             clones_loc = "trimers_raw_tables\\covid_vaccine_new.clones.csv",
                             clones_stats_loc = "trimers_raw_tables\\covid_vaccine_new.clone_stats.csv",
                             metadata_list=["cell_subset", "collection_time_point_relative"],
                             source = source_options,
                             rename_metadata=True,
                             new_metadata_names=["ab_target", "time_point"]
                             )
    
    triemrs = trim_obj.create(subdatasets_list=["ab_target","time_point","subject_id"],  save_csv=True)

> Found cleaned_seqs.csv in the procesed tables folder.
> 'cleaned_seqs.csv' loaded.
> Itirating over sub-datasets (n=30)


100%|██████████| 30/30 [27:40<00:00, 55.35s/dataset]  


> Trimers and unique clones data saved to 'trimers_output' folder.


In [ ]:
####################################################################################
# helper function to assign representitive clone mutation into the germline sequence
def get_mutdf(row) -> str:
    # Relevent columns for the analysis
    clone_id = row["id"]
    mut_tree = row["mutations"]
    sequence = list(row["germline"])

    # Shortcuts name for all mutations types
    type_dict = {"conservative":"conc",
                "nonconservative":"non-cons",
                "synonymous":"syn"}
    
    ## Gettting the clones mutations
    # If clone has mutations
    try:
        # Getting the clone mutations list
        json_tree = json.loads(mut_tree)["regions"]["ALL"]
        json_keys = list(json_tree.keys())

        # List to save the string-json results
        mutations = {}
        
        n = 0
        for k in json_keys:
            key_json = json_tree[k]

            for i in key_json:
                n += 1
                #mutation information
                nt_pos, nt_from, nt_to, count, mut_type =  i["pos"], i["from_nt"], i["to_nt"], i["total"], k
                mut_log = [nt_from + str(nt_pos) + nt_to, count, "unique", type_dict[mut_type]]
                cond_aacheeck = (sequence[nt_pos] == nt_from)
                
                # if the nt positing havent been changed yet OR the mutation has higher count than those that was changed already
                if cond_aacheeck & (nt_pos not in mutations.keys()):
                    mutations[nt_pos] = mut_log
                    sequence[nt_pos] = nt_to

                # if the nt mutation already exsits and it's count is higher than whats been changed already
                elif cond_aacheeck & (nt_pos in mutations.keys()) & (count > mutations[nt_pos][-2]):
                    mutations[nt_pos] = mut_log
                    sequence[nt_pos] = nt_to

                else:
                    raise ValueError(f"nt position {nt_pos} in germline dosen't satisfy: {sequence[nt_pos]} == {nt_from}.")
        
        return mutations, "".join(sequence)

    #if clone dosent have mutations -> returns null values
    except:
        return np.nan, row["germline"]


#############################################################################################
# Merging clones, clone_stats and metadata table to unified dataframe for furthuer processing
def merge_clones(req_metadata : list) -> pd.DataFrame:
    """
    req_metadata -> list of required metadata columns.
    Function that phrases over dataframe and extract the unique mutations for clone id.
    """

    # Defining raw datasets paths
    paths_dict = {"clones":"covid_vaccine_new.clones.csv",
                  "clone_stats":"covid_vaccine_new.clone_stats.csv",
                  "metadata_table":"covid_vaccine_new.sample_metadata.csv"}

    # Importing datasets into the python enviorment
    raw_files = {}
    for f in paths_dict:
        file_path = os.path.join("trimers_raw_tables", paths_dict[f])
        raw_files[f] = pd.read_csv(file_path, index_col=0)

    clone_stats, clones, metadata_table = raw_files["clone_stats"], raw_files["clones"], raw_files["metadata_table"]

    # Cleaning the dataframe and merging required columns
    clone_stats = clone_stats.copy()[["clone_id", "sample_id", "mutations"]]
    clones = clones.copy()[["id", "subject_id", "functional", "germline", "cdr3_aa"]]
    clones_merged = clones.merge(right=clone_stats, left_on="id", right_on="clone_id", how="left")

    # Pivoting the metadatatable for better structure & filtring for needed crows
    metadata_pivot = metadata_table.pivot_table(index="sample_id", columns="key", values="value", aggfunc='first')[req_metadata].reset_index()
    clones_merged = clones_merged.merge(right=metadata_pivot, on="sample_id", how="left")
    clones_merged = clones_merged[clones_merged.sample_id.notnull() & clones_merged.functional == 1] # removing null sample_id rows and non-functional clones

    clones_merged[["mut_log", "mut_sequence"]] = clones_merged.apply(get_mutdf, axis=1, result_type='expand')

    return clones_merged



In [ ]:
test = merge_clones(["collection_time_point_relative","cell_subset"])

In [ ]:
test